# Olist E-Commerce Analysis

## Phase 1: Data Cleaning and RFM Segmentation

This notebook focuses on:

- Data integration
- Data cleaning
- Missing value treatment
- Feature engineering
- Delivery performance analysis
- RFM customer segmentation
- Creation of final cleaned master dataset

## Dataset Loading

Loading all datasets required for customer, order, payment, seller and product analysis.

In [1]:
import pandas as pd
import numpy as np

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [5]:
customers = pd.read_csv('/content/olist_customers_dataset.csv')
orders = pd.read_csv('/content/olist_orders_dataset.csv')
items = pd.read_csv('/content/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/olist_order_payments_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
translation = pd.read_csv('/content/product_category_name_translation.csv')

print("Loaded Successfully")

Loaded Successfully


In [6]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Items:", items.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)

Customers: (99441, 5)
Orders: (70871, 8)
Items: (91624, 7)
Payments: (103886, 5)
Products: (32951, 9)


In [7]:
print("\nCUSTOMERS")
print(customers.isnull().sum())

print("\nORDERS")
print(orders.isnull().sum())

print("\nITEMS")
print(items.isnull().sum())

print("\nPAYMENTS")
print(payments.isnull().sum())

print("\nPRODUCTS")
print(products.isnull().sum())


CUSTOMERS
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

ORDERS
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 118
order_delivered_carrier_date     1255
order_delivered_customer_date    2113
order_estimated_delivery_date       1
dtype: int64

ITEMS
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  1
freight_value          1
dtype: int64

PAYMENTS
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

PRODUCTS
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product

In [8]:
print(customers.columns)
print(orders.columns)
print(items.columns)
print(payments.columns)
print(products.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')


In [9]:
# Fill item table missing values

items['price'] = items['price'].fillna(
    items['price'].median()
)

items['freight_value'] = items['freight_value'].fillna(
    items['freight_value'].median()
)

print("Items cleaned")

Items cleaned


In [10]:
products['product_weight_g'] = products[
    'product_weight_g'
].fillna(
    products['product_weight_g'].median()
)

products['product_length_cm'] = products[
    'product_length_cm'
].fillna(
    products['product_length_cm'].median()
)

products['product_height_cm'] = products[
    'product_height_cm'
].fillna(
    products['product_height_cm'].median()
)

products['product_width_cm'] = products[
    'product_width_cm'
].fillna(
    products['product_width_cm'].median()
)

print("Physical dimensions cleaned")

Physical dimensions cleaned


In [11]:
products = products.merge(
    translation,
    on='product_category_name',
    how='left'
)

print(products.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  \
0             225.0               16.0               10.0              14.0   
1            1000.0     

In [12]:
items['revenue'] = (
    items['price']
    + items['freight_value']
)

items[['price','freight_value','revenue']].head()

,price,freight_value,revenue
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


In [13]:
items['revenue'] = items['price']

In [14]:
master = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

master = master.merge(
    items,
    on='order_id',
    how='left'
)

master = master.merge(
    payments,
    on='order_id',
    how='left'
)

master = master.merge(
    products,
    on='product_id',
    how='left'
)

print(master.shape)

(82365, 32)


In [15]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,18.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,18.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,141.46,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,179.12,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto


In [16]:
master.shape

(82365, 32)

In [18]:
master['product_category_name'] = master[
    'product_category_name'
].fillna('Unknown')

In [19]:
master['order_purchase_timestamp'] = pd.to_datetime(
    master['order_purchase_timestamp']
)

master['order_approved_at'] = pd.to_datetime(
    master['order_approved_at']
)

master['order_delivered_customer_date'] = pd.to_datetime(
    master['order_delivered_customer_date']
)

In [20]:
master['purchase_year'] = (
    master['order_purchase_timestamp'].dt.year
)

master['purchase_month'] = (
    master['order_purchase_timestamp'].dt.month
)

master['purchase_quarter'] = (
    master['order_purchase_timestamp'].dt.quarter
)

master['purchase_day'] = (
    master['order_purchase_timestamp'].dt.day
)

print("Date Features Created")

Date Features Created


In [21]:
master['delivery_days'] = (
    master['order_delivered_customer_date']
    -
    master['order_purchase_timestamp']
).dt.days

master[['delivery_days']].head()

,delivery_days
0,8.0
1,8.0
2,8.0
3,13.0
4,9.0


In [22]:
customer_revenue = master.groupby(
    'customer_unique_id'
)['payment_value'].sum().reset_index()

customer_revenue.columns = [
    'customer_unique_id',
    'total_revenue'
]

customer_revenue.head()

,customer_unique_id,total_revenue
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000f46a3911fa3c0805444483337064,86.22
2,0004aac84e0df4da2b147fca70cf8255,196.89
3,00053a61a98854899e70ed204dd4bafe,838.36
4,0005ef4cd20d2893f0d9fbd94d3c0d97,129.76


In [23]:
customer_orders = master.groupby(
    'customer_unique_id'
)['order_id'].nunique().reset_index()

customer_orders.columns = [
    'customer_unique_id',
    'total_orders'
]

customer_orders.head()

,customer_unique_id,total_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000f46a3911fa3c0805444483337064,1
2,0004aac84e0df4da2b147fca70cf8255,1
3,00053a61a98854899e70ed204dd4bafe,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,1


In [24]:
snapshot_date = master[
    'order_purchase_timestamp'
].max()

In [25]:
rfm = master.groupby(
    'customer_unique_id'
).agg({
    'order_purchase_timestamp':
    lambda x:
    (snapshot_date - x.max()).days,

    'order_id':'nunique',

    'payment_value':'sum'
}).reset_index()

rfm.columns = [
    'customer_unique_id',
    'Recency',
    'Frequency',
    'Monetary'
]

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90
1,0000f46a3911fa3c0805444483337064,585,1,86.22
2,0004aac84e0df4da2b147fca70cf8255,336,1,196.89
3,00053a61a98854899e70ed204dd4bafe,231,1,838.36
4,0005ef4cd20d2893f0d9fbd94d3c0d97,219,1,129.76


In [26]:
rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90
1,0000f46a3911fa3c0805444483337064,585,1,86.22
2,0004aac84e0df4da2b147fca70cf8255,336,1,196.89
3,00053a61a98854899e70ed204dd4bafe,231,1,838.36
4,0005ef4cd20d2893f0d9fbd94d3c0d97,219,1,129.76


In [27]:
master.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'revenue',
       'payment_sequential', 'payment_type', 'payment_installments',
       'payment_value', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'purchase_year', 'purchase_month',
       'purchase_quarter', 'purchase_day', 'delivery_days'],
      dtype='object')

In [28]:
master.isnull().sum().sort_values(ascending=False)

,0
product_category_name_english,15343
product_photos_qty,15330
product_name_lenght,15330
product_description_lenght,15330
order_item_id,14368
product_weight_g,14368
product_height_cm,14368
product_width_cm,14368
product_length_cm,14368
price,14368


In [29]:
num_cols = [
'product_name_lenght',
'product_description_lenght',
'product_photos_qty',
'product_weight_g',
'product_length_cm',
'product_height_cm',
'product_width_cm'
]

for col in num_cols:
    master[col] = master[col].fillna(master[col].median())

In [30]:
cat_cols = [
'product_category_name',
'product_category_name_english'
]

for col in cat_cols:
    master[col] = master[col].fillna('Unknown')

In [31]:
master.isnull().sum().sort_values(ascending=False).head(20)

,0
price,14368
freight_value,14368
revenue,14368
shipping_limit_date,14368
order_item_id,14368
seller_id,14368
product_id,14368
delivery_days,2418
order_delivered_customer_date,2418
order_delivered_carrier_date,1457


In [32]:
master.duplicated().sum()

np.int64(0)

In [33]:
master = master.drop_duplicates()

In [34]:
master.shape

(82365, 37)

In [35]:
customer_spend = master.groupby(
'customer_unique_id'
)['payment_value'].sum().reset_index()

customer_spend.columns = [
'customer_unique_id',
'customer_total_spent'
]

In [36]:
customer_orders = master.groupby(
'customer_unique_id'
)['order_id'].nunique().reset_index()

customer_orders.columns = [
'customer_unique_id',
'customer_total_orders'
]

In [37]:
customer_aov = master.groupby(
'customer_unique_id'
)['payment_value'].mean().reset_index()

customer_aov.columns = [
'customer_unique_id',
'avg_order_value'
]

In [38]:
first_purchase = master.groupby(
'customer_unique_id'
)['order_purchase_timestamp'].min().reset_index()

first_purchase.columns = [
'customer_unique_id',
'first_purchase_date'
]

In [39]:
last_purchase = master.groupby(
'customer_unique_id'
)['order_purchase_timestamp'].max().reset_index()

last_purchase.columns = [
'customer_unique_id',
'last_purchase_date'
]

In [40]:
tenure = first_purchase.merge(
last_purchase,
on='customer_unique_id'
)

tenure['customer_tenure_days'] = (
tenure['last_purchase_date']
-
tenure['first_purchase_date']
).dt.days

In [41]:
product_revenue = master.groupby(
'product_id'
)['payment_value'].sum().reset_index()

product_revenue.columns = [
'product_id',
'product_revenue'
]

In [42]:
product_sales = master.groupby(
'product_id'
)['order_id'].count().reset_index()

product_sales.columns = [
'product_id',
'product_sales_count'
]

In [45]:
rfm['R_score'] = pd.qcut(
rfm['Recency'],
5,
labels=[5,4,3,2,1]
)

In [46]:
rfm['R_score'] = pd.qcut(
rfm['Recency'],
5,
labels=[5,4,3,2,1]
)

In [47]:
rfm['F_score'] = pd.qcut(
rfm['Frequency'].rank(method='first'),
5,
labels=[1,2,3,4,5]
)

In [48]:
rfm['M_score'] = pd.qcut(
rfm['Monetary'],
5,
labels=[1,2,3,4,5]
)

In [49]:
rfm['RFM_Score'] = (
rfm['R_score'].astype(str)
+
rfm['F_score'].astype(str)
+
rfm['M_score'].astype(str)
)

In [50]:
rfm['Segment'] = 'Others'

rfm.loc[
(rfm['R_score']==5) &
(rfm['F_score']==5),
'Segment'
] = 'Champions'

rfm.loc[
(rfm['R_score']>=4) &
(rfm['F_score']>=4),
'Segment'
] = 'Loyal Customers'

rfm.loc[
(rfm['R_score']<=2) &
(rfm['F_score']<=2),
'Segment'
] = 'At Risk'

In [51]:
master[
    master['product_id'].isnull()
]['order_status'].value_counts()

,count
order_status,
delivered,13427
unavailable,458
shipped,192
canceled,192
processing,56
invoiced,38
created,5


In [52]:
master['order_estimated_delivery_date'] = pd.to_datetime(
    master['order_estimated_delivery_date']
)

In [74]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    master[col] = pd.to_datetime(master[col])

In [54]:
master['delivery_delay'] = (
    master['order_delivered_customer_date']
    -
    master['order_estimated_delivery_date']
).dt.days

In [55]:
master[
    master['product_id'].isnull()
]['order_status'].value_counts()

,count
order_status,
delivered,13427
unavailable,458
shipped,192
canceled,192
processing,56
invoiced,38
created,5


In [57]:
rfm['Segment'].value_counts()

,count
Segment,
Others,22153
Loyal Customers,22072
At Risk,22065
Champions,2837


In [58]:
rfm.describe()

,Recency,Frequency,Monetary
count,69127.000000,69127.000000,69127.000000
mean,288.055434,1.025229,203.089262
std,153.623408,0.179046,625.801316
min,0.000000,1.000000,0.000000
25%,163.000000,1.000000,63.435000
50%,269.000000,1.000000,111.300000
75%,398.000000,1.000000,196.260000
max,772.000000,13.000000,109312.640000


In [59]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    master[col] = pd.to_datetime(
        master[col],
        errors='coerce'
    )

In [60]:
master['delivery_delay'] = (
    master['order_delivered_customer_date']
    -
    master['order_estimated_delivery_date']
).dt.days

In [61]:
master['delivery_delay'].head()

,delivery_delay
0,-8.0
1,-8.0
2,-8.0
3,-6.0
4,-18.0


In [62]:
KeyError: 'delivery_delay'

In [63]:
master['delivery_status'] = np.where(
    master['delivery_delay'] > 0,
    'Delayed',
    'On Time'
)

In [64]:
master['delivery_status'].value_counts()

,count
delivery_status,
On Time,77125
Delayed,5240


In [65]:
KeyError: 'Segment'

In [67]:
rfm

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90,4,1,4,414,At Risk
1,0000f46a3911fa3c0805444483337064,585,1,86.22,1,1,2,112,Others
2,0004aac84e0df4da2b147fca70cf8255,336,1,196.89,2,1,4,214,At Risk
3,00053a61a98854899e70ed204dd4bafe,231,1,838.36,3,1,5,315,At Risk
4,0005ef4cd20d2893f0d9fbd94d3c0d97,219,1,129.76,4,1,3,413,At Risk
...,...,...,...,...,...,...,...,...,...
69122,fffbf87b7a1a6fa8b03f081c5f51a201,293,1,167.32,3,5,4,354,Loyal Customers
69123,fffcc512b7dfecaffd80f13614af1d16,189,1,710.70,4,5,5,455,Loyal Customers
69124,fffea47cd6d3cc0a88bd621562a9d061,310,1,84.58,3,5,2,352,Loyal Customers
69125,ffff371b4d645b6ecea244b27531430a,617,1,112.46,1,5,3,153,Loyal Customers


In [69]:
rfm['Segment'].value_counts()

,count
Segment,
Others,22153
Loyal Customers,22072
At Risk,22065
Champions,2837


In [70]:
master['price'] = master['price'].fillna(0)
master['freight_value'] = master['freight_value'].fillna(0)
master['revenue'] = master['revenue'].fillna(0)

In [71]:
master['product_id'] = master['product_id'].fillna('Unknown')
master['seller_id'] = master['seller_id'].fillna('Unknown')

In [72]:
master['order_item_id'] = master['order_item_id'].fillna(0)

In [73]:
import numpy as np
import pandas as pd

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    master[col] = pd.to_datetime(
        master[col],
        errors='coerce'
    )

master['price'] = master['price'].fillna(0)
master['freight_value'] = master['freight_value'].fillna(0)
master['revenue'] = master['revenue'].fillna(0)

master['product_id'] = master['product_id'].fillna('Unknown')
master['seller_id'] = master['seller_id'].fillna('Unknown')
master['order_item_id'] = master['order_item_id'].fillna(0)

master['delivery_delay'] = (
    master['order_delivered_customer_date']
    -
    master['order_estimated_delivery_date']
).dt.days

master['delivery_status'] = np.where(
    master['delivery_delay'] > 0,
    'Delayed',
    'On Time'
)

print("Cleaning Completed")

Cleaning Completed


In [76]:
master.shape

(82365, 39)

In [77]:
master.isnull().sum().sort_values(ascending=False).head(20)

,0
shipping_limit_date,14368
delivery_days,2418
order_delivered_customer_date,2418
delivery_delay,2418
order_delivered_carrier_date,1458
order_approved_at,131
payment_sequential,3
payment_value,3
payment_type,3
payment_installments,3


In [78]:
master.isnull().sum().sort_values(ascending=False).head(20)

,0
shipping_limit_date,14368
delivery_days,2418
order_delivered_customer_date,2418
delivery_delay,2418
order_delivered_carrier_date,1458
order_approved_at,131
payment_sequential,3
payment_value,3
payment_type,3
payment_installments,3


In [79]:
# Numerical columns

master['delivery_days'] = master['delivery_days'].fillna(
    master['delivery_days'].median()
)

master['delivery_delay'] = master['delivery_delay'].fillna(
    master['delivery_delay'].median()
)

master['payment_value'] = master['payment_value'].fillna(
    master['payment_value'].median()
)

master['payment_installments'] = master['payment_installments'].fillna(
    master['payment_installments'].median()
)

master['payment_sequential'] = master['payment_sequential'].fillna(
    master['payment_sequential'].median()
)

In [80]:
date_cols = [
    'shipping_limit_date',
    'order_delivered_customer_date',
    'order_delivered_carrier_date',
    'order_approved_at',
    'order_estimated_delivery_date'
]

for col in date_cols:
    master[col] = master[col].fillna(master[col].mode()[0])

In [81]:
master['payment_type'] = master['payment_type'].fillna('unknown')

In [82]:
master.isnull().sum().sort_values(ascending=False).head(20)

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,0
order_delivered_carrier_date,0
order_delivered_customer_date,0
order_estimated_delivery_date,0
customer_unique_id,0
customer_zip_code_prefix,0


In [84]:
master.to_csv(
    'cleaned_master_dataset.csv',
    index=False
)

print("Final Clean Dataset Saved")

Final Clean Dataset Saved


In [85]:
import os

os.listdir()

['.config',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_products_dataset.csv',
 'olist_order_payments_dataset.csv',
 'product_category_name_translation.csv',
 'olist_customers_dataset.csv',
 'cleaned_master_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'sample_data']

In [87]:
from google.colab import files

files.download('cleaned_master_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Work Done on 09-06-2026 (First Half)

Today I mainly focused on completing the data cleaning and feature engineering part of the Olist E-Commerce dataset.

First, I checked the merged master dataset and verified its shape. The dataset contains around 82,000+ records and all the required columns from orders, customers, products, sellers, payments, and order items datasets were successfully merged.

After merging, I analysed the missing values using:

```python
master.isnull().sum().sort_values(ascending=False)
```

Many columns related to product information, delivery information, and payment details had null values. I handled numerical missing values using median imputation and categorical values using suitable replacements such as "Unknown".

I then checked for duplicate records using:

```python
master.duplicated().sum()
```

The result was 0, which means no duplicate rows were present in the dataset.

Next, I converted all date-related columns into datetime format. During this process, I faced some errors because a few values had inconsistent date formats. After debugging and using proper datetime conversion with error handling, all date columns were successfully converted.

Once the date conversion was completed, I created new delivery-related features.

### delivery_delay

This feature was created by calculating the difference between:

* Actual delivery date
* Estimated delivery date

This helped identify whether an order was delivered early or late.

### delivery_status

Based on delivery_delay, orders were classified as:

* On Time
* Delayed

After that, I worked on customer-level feature engineering.

The following customer features were created:

* customer_total_spent
* customer_total_orders
* avg_order_value
* customer_tenure_days

These features will be useful for CLTV and churn prediction in later stages.

I also created product-level features:

* product_revenue
* product_sales_count

Next, I completed RFM Analysis.

The following metrics were calculated for every customer:

### Recency

Number of days since the customer's last purchase.

### Frequency

Total number of orders placed by the customer.

### Monetary

Total amount spent by the customer.

Based on these values, R, F, and M scores were assigned and combined into a final RFM Score.

Customers were then segmented into different groups:

* Champions
* Loyal Customers
* At Risk
* Others

After completing RFM segmentation, I performed a final validation of the dataset.

I rechecked all missing values and successfully reduced them to zero.

```python
master.isnull().sum().sort_values(ascending=False)
```

Final result:

All important columns contain 0 missing values.

Finally, I saved the cleaned dataset as:

```python
cleaned_master_dataset.csv
```

### Tasks Completed

* Data Merge
* Missing Value Handling
* Duplicate Check
* Date Conversion
* Feature Engineering
* Delivery Analysis
* Customer Feature Creation
* Product Feature Creation
* RFM Analysis
* Customer Segmentation
* Final Dataset Cleaning
* Dataset Export

### Current Progress

Data Loading - Completed

Data Cleaning - Completed

Missing Value Handling - Completed

Feature Engineering - Completed

Data Merge - Completed

RFM Analysis - Completed

Customer Segmentation - Completed

### Next Work

The next phase of the project will focus on:

1. Customer Lifetime Value (CLTV) Analysis
2. Churn Analysis
3. Churn Prediction
4. Customer Clustering
5. Product Analysis
6. Revenue Forecasting
7. Power BI Dashboard


